# ProcessBehavior Tutorial: Fill Weight Analysis

This notebook demonstrates **iterative process behavior analysis** using fill weight data from a 4-lane filling system.

## What We'll Cover

### Part 1: Start Simple - Lane Analysis
- Load and explore data
- Analyze by lane only
- Detect signals and understand lane differences

### Part 2: Go Deeper - Lane + Phase Analysis  
- Add fill cycle phase (up/down arm position)
- Full variance decomposition with residuals
- Main effects and interactions
- Export comprehensive results

## The Data

**Filling System**: 4 lanes, each with up/down fill arm motion
- **pull**: Production sequence (1-100 time points)
- **lane**: Filling lane (1-4)
- **phase**: Fill cycle position (1=down, 2=up)
- **fill_weight**: Target measurement

---

# Part 1: Start Simple - Analyze by Lane

## Step 1: Load and Explore Data

In [ ]:
# Import required libraries
import pandas as pd
from processbehavior import ProcessDataFrame
from processbehavior.signals import SignalConfig
from pathlib import Path

# Find the data file (works from any directory)
data_paths = [
    'processbehavior/datasets/data/FILLWEIGHTDATA_800.csv',  # Running from project root
    '../../processbehavior/datasets/data/FILLWEIGHTDATA_800.csv',  # Running from examples/tom/
]

df = None
for path in data_paths:
    if Path(path).exists():
        df = pd.read_csv(path)
        print(f"✓ Loaded data from: {path}")
        break

if df is None:
    raise FileNotFoundError(
        "Could not find FILLWEIGHTDATA_800.csv. "
        "Make sure you're running this notebook from the project root or examples/tom/ directory."
    )

# Explore the data
print(f"\nDataset: {len(df)} observations")
print(f"  • Pulls (time points): {df['pull'].nunique()}")
print(f"  • Lanes: {sorted(df['lane'].unique())}")
print(f"  • Phases: {sorted(df['phase'].unique())}")
print(f"  • Missing values: {df['fill_weight'].isna().sum()}")

df.head(12)  # Show first 12 rows (3 pulls × 4 lanes)

## Step 2: Create ProcessDataFrame

The `ProcessDataFrame` automatically handles missing values and garbage characters in your data.

In [ ]:
# Create ProcessDataFrame - automatically cleans data
pdf = ProcessDataFrame(df)

print(f"ProcessDataFrame ready with {len(pdf.data)} observations")
print(f"  • Cleaned/removed: {len(df) - len(pdf.data)} rows with missing values")

## Step 3: Analyze by Lane Only

**Question**: Are the 4 lanes performing consistently over time?

We'll group by `lane` only, treating phase as replicates within each lane/pull combination.

In [ ]:
# Run lane-only analysis
analysis_lane = pdf.analyze(
    response_var=pdf.columns.fill_weight,
    grouping_vars=[pdf.columns.lane],  # Group by lane only
    time_var=pdf.columns.pull          # Time sequence
)

result_lane = analysis_lane.calculate()

print("\n" + "=" * 80)
print("LANE ANALYSIS SUMMARY")
print("=" * 80)
print(f"SDS Type: {result_lane.summary['sds']} - {result_lane.summary['sds_description']}")
print(f"Analysis: {result_lane.summary['analysis_type']}")
print(f"Charts Available: {list(result_lane.charts.keys())}")
print(f"Observations: {result_lane.summary['n_observations']}")

### Understanding the Results

**SDS 3**: Partial Replication
- Most (lane × pull) cells have 2 measurements (phase 1 & 2)
- Some cells have missing values → mixed replication
- System uses hybrid variance estimation

**Available Charts**:
- **Xbar**: Average fill weight by lane over time
- **Sbar**: Variation within each lane over time

## Step 4: Visualize Lane Performance

In [ ]:
# Plot Xbar and S charts
fig_lane = result_lane.plot(
    template='processbehavior',
    width=1400,
    height=700,
    highlight_signals=True
)

fig_lane.show()

**Interpretation Tips**:
- **Xbar Chart**: Shows if lane averages are stable over time
- **Sbar Chart**: Shows if lane variation is consistent
- Points beyond limits indicate special causes
- **Interactive**: Hover over points to see details!

## Step 5: Detect Signals

In [ ]:
# Detect signals using Western Electric rules
signals_xbar_lane = result_lane.detect_signals(
    chart='Xbar',
    rules=['rule_1', 'rule_2', 'rule_3', 'rule_4'],
    config=SignalConfig(min_observations=5)  # Lowered for tutorial
)

signals_sbar_lane = result_lane.detect_signals(
    chart='Sbar',
    rules=['rule_1', 'rule_2', 'rule_3', 'rule_4'],
    config=SignalConfig(min_observations=5)
)

print("Signal Detection - Lane Analysis:")
print(f"  Xbar (means): {signals_xbar_lane.count} signals")
print(f"  Sbar (variation): {signals_sbar_lane.count} signals")

if signals_xbar_lane.has_signals:
    print(f"\nXbar Signals:")
    print(signals_xbar_lane.violations[['subgroup', 'value', 'rule', 'description']].head(10))

## Step 6: Check Lane Main Effects

Are some lanes systematically different from others?

In [ ]:
if result_lane.has_effects and 'lane' in result_lane.effects:
    print("Lane Main Effects:")
    print(result_lane.effects['lane'])
    print("\nInterpretation:")
    print("  • 'effect' shows deviation from grand mean")
    print("  • Positive = higher than average")
    print("  • Negative = lower than average")
else:
    print("Main effects not available for this configuration")

---

# Part 2: Go Deeper - Add Phase Analysis

## Step 7: Analyze Lane + Phase

**Question**: Does the fill arm position (up/down) affect fill weight differently across lanes?

Now we'll include **both lane and phase** as grouping variables to see the full picture.

In [ ]:
# Run lane + phase analysis
analysis_full = pdf.analyze(
    response_var=pdf.columns.fill_weight,
    grouping_vars=[pdf.columns.lane, pdf.columns.phase],  # Both factors
    time_var=pdf.columns.pull
)

result_full = analysis_full.calculate()

print("\n" + "=" * 80)
print("LANE + PHASE ANALYSIS SUMMARY")
print("=" * 80)
print(f"SDS Type: {result_full.summary['sds']} - {result_full.summary['sds_description']}")
print(f"Analysis: {result_full.summary['analysis_type']}")
print(f"Charts Available: {list(result_full.charts.keys())}")
print(f"Observations: {result_full.summary['n_observations']}")
print(f"\nCapabilities:")
print(f"  • Residuals: {result_full.has_residuals}")
print(f"  • Main Effects: {result_full.has_effects}")
print(f"  • Interactions: {result_full.has_interactions}")

### Understanding the Change

**SDS 2**: No Replication
- Each (lane × phase × pull) cell has exactly 1 measurement
- No true replicates (phase is now a factor, not a replicate)
- Uses moving range for variance estimation
- Full main effects and interaction analysis available

## Step 8: Visualize Full Analysis

In [ ]:
# Plot all available charts
fig_full = result_full.plot(
    template='processbehavior',
    width=1400,
    height=700,
    highlight_signals=True
)

fig_full.show()

## Step 9: Variance Decomposition (Residuals)

**Wheeler's Residuals** (R1-R5) break down variation into components:
- **R1**: Total deviation from grand mean
- **R2**: Time effects removed
- **R3**: Factor (lane/phase) effects removed
- **R4**: Both time and factor effects removed
- **R5**: Pure error (all systematic effects removed)

In [ ]:
if result_full.has_residuals:
    print("Residuals Available: Yes\n")
    
    # Show first 10 observations with residuals
    display_cols = ['pull', 'lane', 'phase', 'fill_weight', 'R1', 'R2', 'R3', 'R4', 'R5']
    print("Sample Data with Residuals:")
    print(result_full.dataset[display_cols].head(10))
    
    # Calculate variance by residual type
    print("\n\nVariance Breakdown:")
    for col in ['R1', 'R2', 'R3', 'R4', 'R5']:
        var = result_full.residuals[col].var()
        print(f"  {col}: {var:.2f}")
else:
    print("Residuals not available for this SDS")

## Step 10: Main Effects Analysis

### Lane Effects
Do lanes differ systematically?

In [ ]:
if result_full.has_effects:
    if 'lane' in result_full.effects:
        print("Lane Main Effects:")
        print(result_full.effects['lane'])
        print()
    
    if 'phase' in result_full.effects:
        print("\nPhase Main Effects (Up vs Down):")
        print(result_full.effects['phase'])
        print("\nInterpretation:")
        print("  • Phase 1 (down) vs Phase 2 (up) comparison")
        print("  • Shows if arm position affects fill weight")

## Step 11: Interaction Effects

Do lanes behave differently in up vs down positions?

In [ ]:
if result_full.has_interactions:
    print("Interaction Analysis Available: Yes\n")
    
    # Access interaction data from dataset
    if 'lane_x_phase_interaction' in result_full.dataset.columns:
        # Show average by lane-phase combination
        interaction_summary = result_full.dataset.groupby(['lane', 'phase'])['fill_weight'].agg(['mean', 'count'])
        print("Lane × Phase Combinations:")
        print(interaction_summary)
        print("\nInterpretation:")
        print("  • Each cell shows average fill weight for that lane/phase combo")
        print("  • Look for patterns: do some lanes change more between phases?")
    else:
        print("Interaction data structure varies by SDS type")
else:
    print("Interactions not available for this configuration")

## Step 12: Signal Detection (Full Analysis)

In [ ]:
# Detect signals in full analysis
signals_full = result_full.detect_signals(
    chart='Xbar',
    rules=['rule_1', 'rule_2', 'rule_3', 'rule_4'],
    config=SignalConfig(min_observations=5)
)

print(f"Signal Detection - Full Analysis:")
print(f"  • Signals Found: {signals_full.has_signals}")
print(f"  • Total Count: {signals_full.count}")

if signals_full.has_signals:
    print(f"\nSignal Details (first 15):")
    print(signals_full.violations[['subgroup', 'value', 'rule', 'description']].head(15))

## Step 13: Export Comprehensive Results

Save everything to Excel for sharing with the team or further analysis.

In [ ]:
# Export full analysis to Excel
result_full.to_excel(
    'fillweight_analysis_complete.xlsx',
    include_residuals=True,
    include_effects=True,
    include_interactions=True,
    include_full_dataset=True
)

print("✅ Results exported to: fillweight_analysis_complete.xlsx")
print("\nWorkbook includes:")
print("  • Summary sheet with analysis metadata")
print("  • Xbar chart data (subgroup means)")
print("  • Sbar chart data (subgroup variation)")
print("  • VAS Residuals (R1-R5 with original columns)")
print("  • Main Effects (lane and phase)")
print("  • Interactions (lane × phase)")
print("  • Full Dataset (all calculated values)")

---

# Summary: Iterative Analysis Workflow

## What We Learned

### Stage 1: Lane-Only Analysis (SDS 3)
✓ Quick overview of lane performance  
✓ Identify which lanes have issues  
✓ Detect signals in mean and variation  
✓ Phase treated as replicates  

### Stage 2: Lane + Phase Analysis (SDS 2)
✓ Understand effect of fill arm position  
✓ Full variance decomposition (R1-R5)  
✓ Separate main effects for lane and phase  
✓ Interaction analysis (lane × phase)  
✓ Comprehensive Excel export  

## Key Insights

**Iterative Analysis is Powerful**:
1. Start simple (fewer grouping variables)
2. Identify areas needing investigation
3. Add complexity (more grouping variables)
4. Drill down into root causes

**ProcessBehavior Adapts**:
- Automatically detects data structure (SDS)
- Recommends appropriate charts
- Calculates available metrics
- Handles missing values gracefully

## Next Steps

### Explore Further
1. **Stratified Analysis**: Analyze each lane separately with `stratify='lane'`
2. **Time Windows**: Look at specific time ranges for process changes
3. **Custom Rules**: Configure different signal detection rules
4. **Multiple Responses**: Analyze other measurements (weight, volume, etc.)

### Production Use
- Use Wheeler's recommended minimum: 20+ observations for signal detection
- Document special causes when signals occur
- Recalculate limits after process changes
- Share Excel reports with the team

---

**Questions?** This workflow demonstrates how real practitioners use process behavior charts - starting simple and adding detail as needed!